# Week 2.1 | Which DPP-4 inhibitors would you compare?

A structural biologist has time to examine three drug–protein complexes. You have six medicines that all inhibit human dipeptidyl peptidase 4 (DPP-4). **Which three would you choose to represent a range of molecular structures?**

Start by looking at the drawings. A ring or a group of atoms might suggest that two drugs are similar. Molecular descriptors let you compare a specific property, and fingerprints give you a way to compare structural features with a numerical score. At the end, you will decide whether the results support your first choice and look at what a 3D complex can add.

## Learning objectives

By the end of the practical, you should be able to:

- Load the compound table from GitHub, select a molecule by its ID and identify a shared feature in two drawings.
- Compare a molecular graph, SMILES, SELFIES and InChI, and recognise when two SMILES describe the same molecule.
- Use calculated descriptors to compare the properties of two drugs.
- Explain how MACCS and Morgan fingerprints encode structure and interpret Tanimoto scores from RDKit.
- Justify a shortlist of three drugs with evidence from a drawing and a similarity score, and identify which coordinates you need to measure a drug–protein distance.

## Contents

The practical takes **3 hours**, including a 15-minute break.

- **Setup · 10 min** — Open Colab, load the data and read the research question.
- **1. Inspect six drugs · 20 min** — Compare the drawings and make your first shortlist.
- **2. Represent a molecule · 35 min** — Read a graph and three text formats. Check when two records describe the same structure.
- **3. Compare properties · 25 min** — Use descriptors to revisit a pair you chose from the drawings.
- **Break · 15 min**
- **4. Compare fingerprints · 45 min** — Run the similarity search and compare the MACCS and Morgan rankings.
- **5. Make your shortlist · 30 min** — Discuss your choice with another group and look at a bound structure.

Optional exercises at the end cover stereochemistry, a drug–protein distance, a larger similarity search and the radar plot.

### How to work in this notebook

Choose **File → Save a copy in Drive** to keep your work. Run the code cells in order using the play button on the left. When a step asks you to change a value, keep the quotation marks and run that cell again.

For each **Discuss** question, take a moment to write down what you noticed before comparing answers with a partner. Double-click the text cell marked **Your notes** to add your observations.

Each code cell starts with a short explanation. If you want more detail, copy the **Optional GenAI prompt** and the code into your chat. Click **Show code** first if the code is collapsed. Run the cell and check the explanation against what you see.

### Set up Colab

Run the setup cells in order. The compound table is a CSV file stored in the course GitHub repository. You can retrieve the same file each time you open the notebook.

**What the code does.** The install command adds RDKit for reading and drawing molecules and SELFIES for converting molecular strings.

**Optional GenAI prompt**

> Explain what RDKit and SELFIES are used for in this practical.

In [ ]:
# @title Install RDKit and SELFIES { display-mode: "form" }
%pip -q install rdkit selfies

**What the code does.** The imports make functions available for reading tables, drawing molecules and comparing their structures.

**Optional GenAI prompt**

> Explain what pandas and the RDKit modules do in this notebook. Use two sentences.

In [ ]:
# @title Import the packages { display-mode: "form" }
from itertools import combinations

import pandas as pd
import selfies as sf
from IPython.display import display
from rdkit import Chem, DataStructs
from rdkit.Chem import Crippen, Descriptors, Draw, Lipinski, MACCSkeys, rdFingerprintGenerator, rdMolDescriptors

**What the code does.** `data_url` stores the link to the CSV. `read_csv` loads the whole file and uses the compound IDs as row labels. `head(6)` displays the first six rows.

**Optional GenAI prompt**

> Explain how this code reads the CSV from GitHub and labels the rows with compound IDs.

In [ ]:
data_url = "https://raw.githubusercontent.com/tcobanoglu/dadd-course-data/main/data/week2/dpp4_compounds.csv"
collection = pd.read_csv(data_url, index_col="compound_id")
collection = collection.rename(columns={"name": "Drug"})
collection[["Drug", "smiles", "pdb_id"]].head(6)

**What the code does.** The first line keeps the six reference drugs. The second applies `Chem.MolFromSmiles` to each SMILES string to make an RDKit molecule.

**Optional GenAI prompt**

> Explain how this code goes from the SMILES column to six RDKit molecules.

In [ ]:
drugs = collection.loc[collection["reference"]].copy()
molecules = drugs["smiles"].map(Chem.MolFromSmiles)
print(f"{len(molecules)} drugs are ready.")

## 1. Inspect six drugs

In a 2D drawing, an unlabelled corner or line end usually represents a carbon atom. Lines show bonds, and letters such as N, O and F label other elements. Look for rings, branches and labelled atoms as you compare the drawings.

Run the cell for sitagliptin. Then replace `"D05"` with `"D06"` and run it again to display teneligliptin.

**What the code does.** `.loc` uses the ID to select the SMILES and the corresponding RDKit molecule. The last line draws that molecule.

**Optional GenAI prompt**

> Explain how the drug ID selects one molecule and its SMILES.

In [ ]:
drug_id = "D05"
smiles = drugs.loc[drug_id, "smiles"]
molecule = molecules.loc[drug_id]
print(drugs.loc[drug_id, "Drug"], smiles)
Draw.MolToImage(molecule)

**What the code does.** The code puts the six structures in a grid and labels each drawing with the drug name and ID.

**Optional GenAI prompt**

> Explain how this code gets the names for the six drawings.

In [ ]:
# @title Draw the six inhibitors { display-mode: "form" }
labels = [f"{drug_id} {name}" for drug_id, name in drugs["Drug"].items()]
display(Draw.MolsToGridImage(list(molecules), legends=labels, molsPerRow=3, subImgSize=(260, 185)))

**Discuss.**

1. Which two drugs share a feature you can point to in both drawings? Name or describe the feature.
2. Which three drugs would you choose if you wanted the set to include different structural features? Record their IDs and keep your first shortlist for Section 5.

**Your notes**

…

## 2. Represent a molecule

The six drawings in Section 1 came from molecular structures stored as text. Ethanol is small enough to see how the same structure can be written in several ways.

| Representation | What it shows |
|---|---|
| **Molecular graph** | Atoms are points and bonds connect them. You can trace which atoms are neighbours. |
| **SMILES** | A short text description of a structure. `CCO` traces two carbons followed by oxygen. |
| **SELFIES** | A text format that uses bracketed pieces called tokens. For ethanol, these are `[C][C][O]`. |
| **InChI** | The International Chemical Identifier. It gives a standard text identifier for matching chemical records. Look for the molecular formula `C2H6O` in the output. |

**Try it.** Run the cell to see the labelled graph and the usual 2D drawing. The labelled graph shows the non-hydrogen atoms. Trace the two carbons and the oxygen in both drawings, then find them in the SMILES and SELFIES strings.

**What the code does.** RDKit reads `CCO` and draws ethanol. The final lines print the SMILES, SELFIES and InChI for that example.

**Optional GenAI prompt**

> Explain how this cell uses `CCO` to draw ethanol and produce the text formats.

In [ ]:
# @title Show the four representations of ethanol { display-mode: "form" }
smiles = "CCO"
ethanol = Chem.MolFromSmiles(smiles)
labelled_graph = Chem.Mol(ethanol)
for atom in labelled_graph.GetAtoms():
    atom.SetProp("atomLabel", atom.GetSymbol())

display(Draw.MolsToGridImage(
    [labelled_graph, ethanol],
    legends=["Labelled molecular graph", "Standard 2D drawing"],
    molsPerRow=2,
    subImgSize=(250, 160)
))
print("SMILES:", smiles)
print("SELFIES:", sf.encoder(smiles))
print("InChI:", Chem.MolToInchi(ethanol))

### Can two SMILES describe the same molecule?

A SMILES string can also record stereochemistry. RDKit reads a SMILES string as a structure and can write it back as a **canonical SMILES**, a consistent text representation under the same settings.

| SMILES | What it describes |
|---|---|
| `CCO` and `OCC` | Ethanol written from opposite ends |
| `COC` | Dimethyl ether, with oxygen between the two carbons |

**Try it.** Run the cell, then change only `"OCC"` to `"COC"` and run it again. Which comparisons change?

**What the code does.** The first comparison uses the original text. The next two use the structures to compare their canonical SMILES and InChI.

**Optional GenAI prompt**

> Explain what each of the three True or False lines compares. Use `CCO` and `OCC` as examples.

In [ ]:
other_smiles = "OCC"

ethanol = Chem.MolFromSmiles("CCO")
other_molecule = Chem.MolFromSmiles(other_smiles)
print("Same text?", "CCO" == other_smiles)
print("Same canonical SMILES?", Chem.MolToSmiles(ethanol) == Chem.MolToSmiles(other_molecule))
print("Same InChI?", Chem.MolToInchi(ethanol) == Chem.MolToInchi(other_molecule))
display(Draw.MolsToGridImage(
    [ethanol, other_molecule], legends=["CCO", other_smiles], molsPerRow=2, subImgSize=(180, 135)
))

**Discuss.**

1. Which comparisons give the same result for `CCO` and `OCC`? What changes when you use `COC`?
2. Two assays report activity for the same canonical structure. Would you average the two activity values? Which detail about the assays would you check first?

**Your notes**

…

## 3. Compare properties

A **molecular descriptor** is a number that summarises one property. Compare the pair from Section 1 using the values in the table. Vildagliptin (`D01`) and saxagliptin (`D02`) are the worked example.

| Descriptor | What it describes |
|---|---|
| MW (g/mol) | Molecular weight |
| logP | Estimated preference for octanol over water. Higher values indicate a greater preference for octanol. |
| TPSA (Å²) | Estimated polar surface area based on the molecular structure |
| HBD | Number of hydrogen-bond donor sites |
| HBA | Number of hydrogen-bond acceptor sites |

**What the code does.** Each RDKit function calculates one descriptor for every molecule. The code stores the results in a table with one row per drug.

**Optional GenAI prompt**

> Explain how this cell turns the molecule list into a table with five descriptor columns.

In [ ]:
# @title Calculate the five descriptors { display-mode: "form" }
def calculate_properties(structures):
    return pd.DataFrame({
        "MW (g/mol)": structures.map(Descriptors.MolWt),
        "logP": structures.map(Crippen.MolLogP),
        "TPSA (Å²)": structures.map(rdMolDescriptors.CalcTPSA),
        "HBD": structures.map(Lipinski.NumHDonors),
        "HBA": structures.map(Lipinski.NumHAcceptors),
    })

properties = calculate_properties(molecules)

**Try it.** Run the example. Then replace the IDs in quotes with the two drugs you found similar. Change one ID at a time. Read the **same column** for both drugs.

**What the code does.** The last line places the drug names beside the calculated values and displays the two rows you selected.

**Optional GenAI prompt**

> Explain how changing a drug ID changes the rows shown in the table.

In [ ]:
first_drug = "D01"
second_drug = "D02"

pair = [first_drug, second_drug]
drugs[["Drug"]].join(properties).loc[pair].round(2)

**Discuss.**

1. Which descriptor supports your choice of a similar pair? Give the value for each drug.
2. What structural difference can you see in the drawings that the five descriptors leave out?

**Your notes**

…

### Break · 15 min

## 4. Compare fingerprints

A molecular **fingerprint** records features of a structure. In the binary fingerprints used here, a bit is on (1) when the fingerprint records a feature.

| Term | How it works |
|---|---|
| Substructure-based fingerprint | Records patterns within a molecule. A common approach is to check every molecule against the same list of predefined patterns. These are called structural keys. |
| MACCS fingerprint | A structural-key fingerprint. The public set defines 166 keys. Each key checks a predefined feature, such as a pattern of bonded atoms. |
| Circular fingerprint | Builds features around each atom by following bonds outward. The radius tells you how many bonds away to look. |
| Morgan fingerprint | A circular fingerprint. Here we look up to two bonds from each atom (radius 2) and store the resulting features in 2,048 bits. The settings used here ignore stereochemistry. |

For example, start at the oxygen in ethanol. One bond takes you to the adjacent carbon. Two bonds take you to the other carbon.

**Tanimoto similarity** is a score from 0 to 1. A higher score means the two fingerprints share a larger fraction of the bits that are on in either fingerprint. RDKit calculates the scores in this practical.

### Check what happens when the SMILES changes

**Try it.** Run the two cells below. Then change `"OCC"` to `"COC"`. Does the MACCS bit for an oxygen bonded to hydrogen change? What happens to the two similarity scores?

RDKit calculates both fingerprints from the structure it reads from SMILES. A score of 1 means the on-bits match for that fingerprint. Different molecules can share the same fingerprint.

**What the code does.** This cell makes MACCS and Morgan fingerprints. It also defines the function that compares ethanol with a second structure.

**Optional GenAI prompt**

> Explain how the function compares the same pair of structures with two fingerprints.

In [ ]:
# @title Prepare MACCS and Morgan fingerprints { display-mode: "form" }
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
maccs_fingerprints = molecules.map(MACCSkeys.GenMACCSKeys)
fingerprints = molecules.map(morgan.GetFingerprint)

def show_smiles_scores(smiles):
    ethanol = Chem.MolFromSmiles("CCO")
    other = Chem.MolFromSmiles(smiles)
    ethanol_maccs = MACCSkeys.GenMACCSKeys(ethanol)
    other_maccs = MACCSkeys.GenMACCSKeys(other)
    print("Same canonical SMILES?", Chem.MolToSmiles(ethanol) == Chem.MolToSmiles(other))
    print(f"MACCS key 139 (oxygen bonded to H): CCO = {int(ethanol_maccs[139])}, {smiles} = {int(other_maccs[139])}")
    maccs_score = DataStructs.TanimotoSimilarity(ethanol_maccs, other_maccs)
    morgan_score = DataStructs.TanimotoSimilarity(morgan.GetFingerprint(ethanol), morgan.GetFingerprint(other))
    print(f"MACCS similarity: {maccs_score:.3f}")
    print(f"Morgan similarity: {morgan_score:.3f}")

**What the code does.** The SMILES in the first line selects the structure to compare with ethanol. The function prints one MACCS bit and the two similarity scores.

**Optional GenAI prompt**

> Explain what the output tells us about ethanol and the molecule entered here.

In [ ]:
comparison_smiles = "OCC"
show_smiles_scores(comparison_smiles)

**Discuss.** Use the drawings and the two sets of results to explain why changing a SMILES can leave the fingerprint as it was or change it.

**Your notes**

…

### Find the closest drug with MACCS and Morgan

A **reference drug** is the one you compare with the others. Start with sitagliptin (`D05`). Look back at the drawings and predict which drug will rank first. Run both cells below. Each table ranks the other five drugs. The reference drug and Tanimoto calculation are the same in both tables.

**What the code does.** The code removes the reference drug from the candidates, calculates similarity scores and sorts the matches from highest to lowest.

**Optional GenAI prompt**

> Explain how the code builds the rankings. Use three short steps.

In [ ]:
# @title Prepare the similarity search { display-mode: "form" }
def rank_neighbours(reference_id, library, library_fingerprints):
    candidates = library_fingerprints.drop(reference_id)
    scores = DataStructs.BulkTanimotoSimilarity(
        library_fingerprints.loc[reference_id], list(candidates)
    )
    matches = library.loc[candidates.index, ["Drug"]].copy()
    matches["Tanimoto similarity"] = scores
    return matches.sort_values("Tanimoto similarity", ascending=False, kind="stable")

def show_neighbours(reference_id):
    maccs_ranking = rank_neighbours(reference_id, drugs, maccs_fingerprints)
    morgan_ranking = rank_neighbours(reference_id, drugs, fingerprints)
    print("MACCS ranking")
    display(maccs_ranking.round(4))
    print("Morgan ranking")
    display(morgan_ranking.round(4))
    maccs_match = maccs_ranking.index[0]
    morgan_match = morgan_ranking.index[0]
    display(Draw.MolsToGridImage(
        list(molecules.loc[[reference_id, maccs_match, morgan_match]]),
        legends=[
            f"Reference: {drugs.at[reference_id, 'Drug']}",
            f"MACCS: {drugs.at[maccs_match, 'Drug']}",
            f"Morgan: {drugs.at[morgan_match, 'Drug']}",
        ],
        molsPerRow=3,
        subImgSize=(250, 175)
    ))

**What the code does.** The drug ID selects the reference molecule. The output shows a MACCS ranking, a Morgan ranking and drawings of the top matches.

**Optional GenAI prompt**

> Explain how the reference ID changes the rankings and drawings.

In [ ]:
reference_drug = "D05"
show_neighbours(reference_drug)

**Try it.** Replace `"D05"` with `"D06"` to use teneligliptin as the reference. Run the ranking cell again.

**Discuss.**

1. Which drug ranks first with each fingerprint? Find a feature in the drawings that teneligliptin shares with one of those drugs.
2. Look at the two highest Morgan scores. How far apart are they? How would you describe that difference to someone choosing which complex to examine?

**Your notes**

…

## 5. Make your shortlist

The ranking in Section 4 compared each drug with a single reference. Now compare **all three pairs** within your shortlist using the Morgan fingerprints. For three drugs A, B and C, those pairs are A–B, A–C and B–C.

**Try it.** Run both cells. Then replace the three quoted IDs in the second cell with your first shortlist. The table shows the most similar pair at the top. If you want more variety, try replacing one member of that pair and check whether the highest score goes down.

**What the code does.** The function forms each pair once, calculates its Morgan-based Tanimoto score and puts the highest score first.

**Optional GenAI prompt**

> Explain how this code forms and scores the three pairs.

In [ ]:
# @title Prepare the shortlist comparison { display-mode: "form" }
def show_shortlist(selected_drugs):
    if len(selected_drugs) != 3 or len(set(selected_drugs)) != 3 or not set(selected_drugs) <= set(drugs.index):
        print("Choose three different IDs from D01 to D06.")
    else:
        comparisons = [
            (f"{left} {drugs.at[left, 'Drug']}", f"{right} {drugs.at[right, 'Drug']}",
             DataStructs.TanimotoSimilarity(fingerprints.loc[left], fingerprints.loc[right]))
            for left, right in combinations(selected_drugs, 2)
        ]
        pair_scores = pd.DataFrame(comparisons, columns=["First drug", "Second drug", "Tanimoto similarity"])
        display(pair_scores.sort_values("Tanimoto similarity", ascending=False).round(3))

**What the code does.** The list gives the three drug IDs to compare. The call below it runs the pair comparison using that list.

**Optional GenAI prompt**

> Show which pairs the code compares for these three IDs.

In [ ]:
selected_drugs = ["D01", "D02", "D05"]
show_shortlist(selected_drugs)

**Discuss with another group.** Compare the most similar pair in each shortlist. Would their drawings or scores make you replace one of your choices? Explain what you would gain by making the change.

Record your final three IDs. Support your selection with one visible structural difference and one similarity score.

**Your notes**

…

### From a 2D drawing to a bound structure

Open the **3D View** of [PDB entry 1X70](https://www.rcsb.org/structure/1X70). Look for sitagliptin, labelled `715`, inside DPP-4. Its **bound pose** describes how the ligand is positioned and oriented in the protein.

**Discuss.**

1. Which representation would you use to find sitagliptin’s rings? What information would you need to measure the distance from a sitagliptin atom to a DPP-4 atom?
2. Both course files end in `.sdf`. What detail in the file or its source would tell you which coordinates came from the protein–ligand complex? Compare [sitagliptin_2d.sdf](https://github.com/tcobanoglu/dadd-course-data/blob/main/data/week2/sitagliptin_2d.sdf) and [sitagliptin_bound.sdf](https://github.com/tcobanoglu/dadd-course-data/blob/main/data/week2/sitagliptin_bound.sdf).
3. If two records have the same canonical SMILES, what would you check before saying that they show the same pose?

**For Week 2.2.** Bring your shortlist and one question about how the selected drugs bind to DPP-4.

**Your notes**

…

## Optional extensions

### Can a fingerprint tell mirror images apart?

These two SMILES describe mirror-image forms of lactic acid. The atoms connect in the same order. The `@` and `@@` labels indicate opposite configurations at the same carbon in these two strings.

**Try it.** Run the cell. Do the canonical SMILES match? Do the fingerprints match with the settings we used? What changes when Morgan includes stereochemistry?

**What the code does.** The code compares the two lactic acid structures with MACCS, with Morgan and with Morgan set to include stereochemistry.

**Optional GenAI prompt**

> Explain what changes when Morgan includes stereochemistry.

In [ ]:
# @title Compare the two forms of lactic acid { display-mode: "form" }
first = Chem.MolFromSmiles("C[C@H](O)C(=O)O")
second = Chem.MolFromSmiles("C[C@@H](O)C(=O)O")
morgan_with_stereo = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=2048, includeChirality=True
)
print("Same canonical SMILES?", Chem.MolToSmiles(first) == Chem.MolToSmiles(second))
for label, make_fingerprint in [
    ("MACCS", MACCSkeys.GenMACCSKeys),
    ("Morgan", morgan.GetFingerprint),
    ("Morgan with stereochemistry", morgan_with_stereo.GetFingerprint),
]:
    score = DataStructs.TanimotoSimilarity(make_fingerprint(first), make_fingerprint(second))
    print(f"{label} similarity: {score:.3f}")

### Measure a distance in the 1X70 complex

In the X-ray structure of DPP-4 with sitagliptin, the ligand and protein atoms have coordinates in the same 3D frame. The code below uses coordinates for three atoms from chain A of the deposited model. `N20` is an atom in sitagliptin. `OE1` and `OE2` are the two side-chain oxygen atoms in the protein residue Glu205.

**Try it.** Run the cell. Then replace `"OE1"` with `"OE2"` and run it again. Which oxygen is closer to the drug atom? Why would a 2D drawing of sitagliptin alone be insufficient for this measurement?

[PubChem 2D SDF](https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/4369359/SDF?record_type=2d) · [Bound ligand SDF](https://models.rcsb.org/v1/1x70/ligand?auth_asym_id=A&label_comp_id=715&encoding=sdf) · [Full 1X70 PDB file](https://files.rcsb.org/download/1X70.pdb)

**What the code does.** The first line selects an oxygen atom. The code calculates the distance between that oxygen and the ligand nitrogen using their 3D coordinates.

**Optional GenAI prompt**

> Explain how the two sets of coordinates give an atom-to-atom distance.

In [ ]:
oxygen = "OE1"

from math import dist

# Coordinates in ångströms from 1X70, chain A.
ligand_n20 = (38.203, 51.846, 34.353)
glu205 = {'OE1': (38.443, 56.65, 35.021), 'OE2': (38.201, 54.481, 35.372)}

distance = dist(ligand_n20, glu205[oxygen])
print(f"N20 to Glu205 {oxygen}: {distance:.2f} Å")

### Search the larger collection

Your first ranking used only the six reference drugs. The course collection contains 22 more ligands from DPP-4 structures. Compare this search’s top score with the top Morgan score for sitagliptin in Section 4. Which ligand gives the higher score?

[Week 2 course data and provenance](https://github.com/tcobanoglu/dadd-course-data/tree/main/data/week2)

**What the code does.** The code reuses the full GitHub table, calculates Morgan fingerprints and displays the five closest matches to sitagliptin.

**Optional GenAI prompt**

> Explain how this search goes from all 28 compounds to five ranked matches.

In [ ]:
# @title Search all 28 compounds { display-mode: "form" }
collection_molecules = collection["smiles"].map(Chem.MolFromSmiles)
collection_fingerprints = collection_molecules.map(morgan.GetFingerprint)
rank_neighbours("D05", collection, collection_fingerprints).head(5).round(3)

### Read a radar plot

The table in Section 3 shows the original values. A radar plot puts these descriptors on separate spokes and scales each one from 0 to 1 using the smallest and largest values in the 28-compound collection. Run **Search all 28 compounds** first. This plot uses the pair you chose in Section 3.

**Discuss.** Suppose a new compound has a higher TPSA than all 28 compounds. What would happen to your two drugs’ positions on the TPSA spoke after rescaling? What TPSA values would still appear in the table?

**What the code does.** The code scales each descriptor using its range across all 28 compounds and plots the scaled values for your selected pair.

**Optional GenAI prompt**

> Explain what 0 and 1 mean on one spoke of the radar plot.

In [ ]:
# @title Plot the five scaled descriptors { display-mode: "form" }
import matplotlib.pyplot as plt
import numpy as np

collection_properties = calculate_properties(collection_molecules)
minimum = collection_properties.min()
maximum = collection_properties.max()
scaled_pair = (properties.loc[pair] - minimum) / (maximum - minimum)

angles = np.linspace(0, 2 * np.pi, len(properties.columns), endpoint=False)
closed_angles = np.append(angles, angles[0])
figure, axis = plt.subplots(figsize=(5.5, 4.5), subplot_kw={"projection": "polar"}, layout="constrained")
for drug_id, values in scaled_pair.iterrows():
    axis.plot(closed_angles, np.append(values, values.iloc[0]), marker="o", label=drugs.at[drug_id, "Drug"])
axis.set_xticks(angles, ["MW", "logP", "TPSA", "HBD", "HBA"])
axis.set_ylim(0, 1.05)
axis.set_title("Descriptor ranges across 28 compounds", pad=20)
axis.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
plt.show()

## References

- TeachOpenCADD [T033 Molecular representations](https://projects.volkamerlab.org/teachopencadd/talktorials/T033_molecular_representations.html), [T002 Molecular filtering](https://projects.volkamerlab.org/teachopencadd/talktorials/T002_compound_adme.html), [T004 Compound similarity](https://projects.volkamerlab.org/teachopencadd/talktorials/T004_compound_similarity.html) and [T008 Protein Data Bank](https://projects.volkamerlab.org/teachopencadd/talktorials/T008_query_pdb.html) informed the order of topics.
- [RDKit Getting Started](https://www.rdkit.org/docs/GettingStartedInPython.html) covers drawing molecules, calculating descriptors and working with fingerprints.
- [SELFIES documentation](https://selfies.readthedocs.io/en/latest/) and the [IUPAC definition of InChI](https://goldbook.iupac.org/terms/view/11456) describe two text formats for chemical structures.
- The six reference compounds and their identifiers come from the [Week 2 course table](https://github.com/tcobanoglu/dadd-course-data/blob/main/data/week2/dpp4_compounds.csv). The experimental example comes from [RCSB PDB entry 1X70](https://www.rcsb.org/structure/1X70).